In [0]:
from pyspark.sql.functions import col, trim, lower, upper, current_timestamp, translate, regexp_replace, to_date, to_timestamp

# 1. Parámetros
dbutils.widgets.text("env", "dev", "Ambiente")
dbutils.widgets.text("tabla", "movimientos_inventario", "Nombre de la tabla")
dbutils.widgets.text("pk", "id_movimiento", "Llave Primaria")

ambiente = dbutils.widgets.get("env")
nombre_tabla = dbutils.widgets.get("tabla")

tabla_origen = f"ferreteria_{ambiente}.bronze.{nombre_tabla}"
tabla_destino = f"ferreteria_{ambiente}.silver.{nombre_tabla}"

print(f"Limpiando tabla: {tabla_origen}")

# 2. Lectura de la tabla cruda (Bronce)
df = spark.read.table(tabla_origen)

# 3. LIMPIEZA UNIVERSAL DINÁMICA

# A. Eliminar columnas técnicas de Bronce
if "_rescued_data" in df.columns:
    df = df.drop("_rescued_data")

# B. Convertir todos los nombres de columnas a minúsculas
df = df.select([col(c).alias(c.lower()) for c in df.columns])

# C. Identificar columnas de texto (String)
columnas_texto = [f.name for f in df.schema.fields if f.dataType.typeName() == 'string']

for c in columnas_texto:
    # 1. Limpiar espacios y ajustar mayúsculas/minúsculas
    if c == 'rfc':
        df = df.withColumn(c, upper(trim(col(c)))) # RFC siempre en mayúsculas
    else:
        df = df.withColumn(c, lower(trim(col(c)))) # Lo demás en minúsculas
    
    # 2. Quitar acentos
    df = df.withColumn(c, translate(col(c), 'áéíóúü', 'aeiouu'))
    
    # 3. Limpiar símbolos raros (Ojo: agregué A-Z para no borrar el RFC)
    df = df.withColumn(c, regexp_replace(col(c), r'[^a-zA-Z0-9\s\.\-@_]', ''))

# D. Tratamiento de Fechas y Nulos
for c in df.columns:
    if "fecha" in c:
        if "hora" in c or "timestamp" in c:
            df = df.withColumn(c, to_timestamp(col(c)))
        else:
            df = df.withColumn(c, to_date(col(c)))
    
    if c in columnas_texto:
        df = df.fillna('n/d', subset=[c])

# E. Eliminar Duplicados completos
df = df.dropDuplicates()

# F. Columna de Auditoría
df = df.withColumn("fecha_carga_plata", current_timestamp())

# 4. GUARDADO EN UNITY CATALOG (Lógica de Upsert / MERGE)
from delta.tables import DeltaTable

# Jalamos la llave primaria del nuevo widget (asegúrate de agregarlo arriba)
llave_primaria = dbutils.widgets.get("pk")

# Verificamos si la tabla ya existe en la capa Silver
tabla_existe = spark.catalog.tableExists(tabla_destino)

if not tabla_existe:
    print(f"[*] La tabla {tabla_destino} no existe. Ejecutando carga inicial (Overwrite)...")
    df.write.format("delta").mode("overwrite").saveAsTable(tabla_destino)
    print("[OK] Tabla creada con éxito en la capa Plata.")
else:
    print(f"[*] La tabla {tabla_destino} ya existe. Ejecutando MERGE (Upsert)...")
    tabla_delta = DeltaTable.forName(spark, tabla_destino)
    
    # El motor de actualización dinámica
    (tabla_delta.alias("destino")
     .merge(
         df.alias("origen"),
         f"destino.{llave_primaria} = origen.{llave_primaria}"
     )
     .whenMatchedUpdateAll()     # Si el ID ya existe, actualiza todos los campos con la nueva info
     .whenNotMatchedInsertAll()  # Si el ID es nuevo, inserta el registro completo
     .execute()
    )
    print("[OK] Datos fusionados (Merged) con éxito.")